# PDS 37 with VLTI/GRAVITY: binary fits epoch by epoch

The same binary analysis is run on each GRAVITY exposure whose closure phases are clean enough to constrain a
binary on its own (listed in `data/gravity_good_files.txt`, see `data/README.md`). Fitting the epochs separately
(rather than all together) shows whether the companion's position is consistent from one observation to the next.

For each epoch:
1. **grid search** for the companion position (K band, binned ×20),
2. **refine** all parameters, including the two disk diameters,
3. **bootstrap** the fit (resampling the data) to get realistic uncertainties,
4. fit **Brackett-γ line emission** for each star at full spectral resolution, starting from the continuum solution.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pmoired

sys.path.insert(0, str(Path.cwd().parent))  # make the local `pds37` package importable
from pds37 import config

plt.rcParams["figure.dpi"] = 80
from IPython.display import Markdown, display
from pds37 import oifits
from pds37.binary import (binary_grid_search, with_brg_lines, image_window,
                          separation_and_pa, fit_table, BRG_LINE_PRIORS)

N_BOOTSTRAP = 500  # number of bootstrap resamplings per epoch

epochs = oifits.summarize_files(config.gravity_good_files())
epochs = epochs.sort_values("mjd_start").reset_index(drop=True)
epochs

## Fitting functions

In [ ]:
def fit_continuum(path):
    """Grid search, refine and bootstrap a two-uniform-disk binary in the K-band continuum."""
    oi = pmoired.OI(str(path), insname=config.GRAVITY_INSNAME, binning=20, verbose=False)
    oi.setupFit({"obs": ["|V|", "T3PHI"], "wl ranges": [config.K_BAND]})

    start = binary_grid_search(oi, max_sep=40, step=3)
    oi.showGrid(interpolate=True, logV=1, cmap="gist_stern")

    oi.doFit(start, doNotFit=[], verbose=0)
    oi.show(**image_window(oi.bestfit["best"]))

    oi.bootstrapFit(Nfits=N_BOOTSTRAP, verbose=0)
    oi.showBootstrap()
    return {"best": dict(oi.boot["best"]), "uncer": dict(oi.boot["uncer"]),
            "chi2": oi.bestfit["chi2"]}


def fit_brg(path, continuum):
    """Fit Brackett-gamma emission on both stars around 2.166 µm at full resolution."""
    oi = pmoired.OI(str(path), insname=config.GRAVITY_INSNAME, verbose=False)
    oi.setupFit({"obs": ["|V|", "T3PHI", "NFLUX", "DPHI"],
                 "wl ranges": [config.BRG_WINDOW],
                 "NFLUX order": 2})
    oi.doFit(with_brg_lines(continuum["best"]), prior=BRG_LINE_PRIORS, verbose=0)
    oi.show(**image_window(oi.bestfit["best"]))
    return {"best": dict(oi.bestfit["best"]), "uncer": dict(oi.bestfit["uncer"]),
            "chi2": oi.bestfit["chi2"]}

## Run every epoch

In [ ]:
continuum, brg = {}, {}
for name, date in zip(epochs.file, epochs["date_obs (UT)"]):
    display(Markdown(f"### {date[:10]} — `{name}`"))
    path = config.gravity_path(name)
    continuum[date[:10]] = fit_continuum(path)
    brg[date[:10]] = fit_brg(path, continuum[date[:10]])
    plt.show()

## Results

Continuum binary parameters (median and 1σ from the bootstrap). `A,ud` and `B,ud` are uniform-disk diameters
in mas, `B,f` the companion/primary flux ratio, and `B,x`/`B,y` the companion offset East/North in mas.

In [ ]:
summary = fit_table(continuum, ["A,ud", "B,ud", "B,f", "B,x", "B,y"])
summary["reduced χ²"] = [round(c["chi2"], 2) for c in continuum.values()]

for date, c in continuum.items():
    sep, sep_err, pa, pa_err = separation_and_pa(c["best"]["B,x"], c["best"]["B,y"],
                                                 c["uncer"]["B,x"], c["uncer"]["B,y"])
    summary.loc[date, "separation (mas)"] = f"{sep:.2f} ± {sep_err:.2f}"
    summary.loc[date, "PA (deg E of N)"] = f"{pa:.1f} ± {pa_err:.1f}"
summary

Brackett-γ line parameters: `line_brg_f` is the line flux relative to each star's continuum (positive = emission), `line_brg_wl0` the central wavelength (µm) and `line_brg_lorentzian` the width (nm).

In [ ]:
fit_table(brg, ["A,line_brg_f", "A,line_brg_wl0", "A,line_brg_lorentzian",
               "B,line_brg_f", "B,line_brg_wl0", "B,line_brg_lorentzian"])

### Companion position at each epoch

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(0, 0, "k*", ms=14, label="primary (A)")
for date, c in continuum.items():
    b, u = c["best"], c["uncer"]
    ax.errorbar(b["B,x"], b["B,y"], xerr=u["B,x"], yerr=u["B,y"], fmt="o", capsize=3, label=date)
lim = 1.2 * max(np.hypot(c["best"]["B,x"], c["best"]["B,y"]) for c in continuum.values())
ax.set(xlim=(lim, -lim), ylim=(-lim, lim), aspect="equal",
       xlabel="ΔRA, East → left (mas)", ylabel="ΔDec, North ↑ (mas)",
       title="Fitted companion position (bootstrap 1σ)")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="best");

**Reading the plot:** if the epochs disagree by much more than their error bars, the single-epoch solutions are
not all trustworthy (sparse uv coverage on a single snapshot can make several positions fit almost equally well),
or the model is missing structure such as extended circumstellar emission. Either way it is a warning against
combining all epochs into one static model without checking, which is what
`04_gravity_joint_fit_all_epochs.ipynb` does.